# 模型持久化、导出与一致性验证

## 学习目标

能够区分 state_dict、训练检查点和 TorchScript，并验证保存前后的输出一致。


## 概念模型与执行路径

state_dict 依赖 Python 模型定义，训练检查点还包含优化状态，TorchScript 保存可独立加载的执行图。导出成功只是格式正确，仍需用代表性输入验证数值和动态行为。


### 实验 1


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "07-deep-learning/pytorch",
]
PYTORCH_ROOT = next(path for path in candidates if (path / "common").exists())
if str(PYTORCH_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTORCH_ROOT))
print("course root:", PYTORCH_ROOT)


### 实验 2


In [ ]:
import tempfile
import torch
from common.models import ImageClassifier
model = ImageClassifier().eval()
example = torch.randn(2, 1, 28, 28)
with torch.inference_mode():
    eager_output = model(example)
    scripted = torch.jit.trace(model, example)
print(eager_output.shape)


### 实验 3


In [ ]:
with tempfile.TemporaryDirectory() as directory:
    destination = Path(directory) / "model.pt"
    scripted.save(str(destination))
    loaded = torch.jit.load(str(destination)).eval()
    with torch.inference_mode():
        loaded_output = loaded(example)
    torch.testing.assert_close(eager_output, loaded_output)
    print("round-trip bytes:", destination.stat().st_size)


### 实验 4


In [ ]:
different_batch = torch.randn(5, 1, 28, 28)
with torch.inference_mode():
    print("different batch shape:", loaded(different_batch).shape)


### 实验 5


In [ ]:
# python 07-deep-learning/pytorch/examples/export_model.py --quick


## 底层机制

Tracing 记录示例输入经过的运算路径，数据依赖的 Python 控制流可能被固化。导出验证应覆盖不同 batch 和边界输入。生产部署还要固定预处理、类别映射和版本。


## 检查点

为什么只比较文件是否生成不足以证明导出正确？state_dict 与 TorchScript 各自依赖什么？


## 试一试

导出后用 batch size 1 和 5 验证输出；再加入依赖输入值的 Python if，观察 tracing 警告或行为差异。


## 常见错误与调试

导出时未 eval、遗漏预处理规范、只测试一个输入、把训练检查点当成可直接部署模型。
